In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1BJxkUqnncRvIWFd3IuJimXGwDL29WJgu", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/04_01_intro.mp3"))

# 🚀 CI/CD Integration — Automating Claude Code in Pipelines

In this notebook, we will build a working simulation of Claude Code's CI/CD integration. You will construct CLI argument parsers, JSON schema validators, and a mock PR review pipeline — all the pieces tested on Domain 3 of the architect exam.

By the end, you will be able to:
- Use the `-p` flag for non-interactive mode in CI pipelines
- Structure output with `--output-format json` and `--json-schema`
- Validate structured review output against JSON Schema
- Design CI workflows with session isolation and deduplication
- Configure CLAUDE.md for automated review context

# 🤖 AI Teaching Assistant

Need help with this notebook? Open the **AI Teaching Assistant** — it has already read this entire notebook and can help with concepts, code, and exercises.

**[👉 Open AI Teaching Assistant](https://pods.vizuara.ai/courses/claude-certified-architect/claude-code-workflows/practice/4/assistant)**

*Tip: Open it in a separate tab and work through this notebook side-by-side.*


In [ ]:
#@title 🎧 Listen: Why Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_02_why_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Your team spends 2 hours per PR on code review. A senior engineer reads every diff, checks for type safety, validates error handling, and verifies test coverage. This is valuable but slow.

Claude Code can automate the first pass. It reads the diff, applies your team's conventions (from CLAUDE.md), and posts structured feedback directly on the PR. The human reviewer then focuses on architecture and design — the parts that require human judgment.

But CI/CD integration has subtleties: structured output schemas, session isolation to avoid self-review bias, and deduplication of prior findings. The exam tests all of these.

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_03_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
import json
import re
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
from datetime import datetime

In [ ]:
#@title 🎧 Listen: P Flag
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_04_p_flag.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition — The -p Flag

Think of `-p` (or `--print`) as putting Claude Code in "autopilot mode." In interactive mode, Claude asks for confirmation before running tools, waits for your input, and carries on a conversation. In `-p` mode, it:

1. Reads the prompt from the command line or stdin
2. Executes without human interaction
3. Prints the result to stdout
4. Exits

This is what makes Claude Code usable in a CI pipeline — there is no human at the terminal to approve tool calls.

In [ ]:
#@title 🎧 Listen: Cli Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_05_cli_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
@dataclass
class CLIInvocation:
    """Simulates a Claude Code CLI invocation."""
    prompt: str
    output_format: str = "text"     # "text" or "json"
    json_schema: Optional[dict] = None
    print_mode: bool = False        # -p flag
    model: str = "claude-sonnet-4-20250514"

    def to_command(self) -> str:
        """Generate the equivalent CLI command."""
        parts = ["claude"]

        if self.print_mode:
            parts.append("-p")

        parts.append(f'"{self.prompt}"')

        if self.output_format == "json":
            parts.append("--output-format json")

        if self.json_schema:
            schema_str = json.dumps(self.json_schema)
            parts.append(f"--json-schema '{schema_str}'")

        return " \\
  ".join(parts)


# Example: Simple text review (not ideal for CI)
text_review = CLIInvocation(
    prompt="Review the changes in this PR for bugs",
    print_mode=True,
)

# Example: Structured JSON review (ideal for CI)
json_review = CLIInvocation(
    prompt="Review this PR for issues",
    print_mode=True,
    output_format="json",
    json_schema={
        "type": "object",
        "properties": {
            "issues": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "file": {"type": "string"},
                        "line": {"type": "integer"},
                        "severity": {"type": "string", "enum": ["critical", "warning", "info"]},
                        "description": {"type": "string"},
                        "suggestion": {"type": "string"}
                    },
                    "required": ["file", "line", "severity", "description"]
                }
            },
            "summary": {"type": "string"},
            "approve": {"type": "boolean"}
        },
        "required": ["issues", "summary", "approve"]
    }
)

print("❌ Text mode — hard to parse in CI:")
print(text_review.to_command())
print()
print("✅ Structured JSON mode — machine-parseable:")
print(json_review.to_command())

In [ ]:
#@title 🎧 Listen: Schema Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_06_schema_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. JSON Schema for Structured Review Output

The `--json-schema` flag is what makes CI integration practical. Without it, Claude returns freeform text that your pipeline cannot reliably parse. With it, Claude returns structured JSON that your pipeline can process automatically.

Let us build a schema validator to understand how this works.

In [ ]:
#@title 🎧 Listen: Schema Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_07_schema_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def validate_against_schema(data: dict, schema: dict) -> List[str]:
    """
    Simple JSON Schema validator (subset of JSON Schema spec).
    Returns a list of validation errors (empty = valid).
    """
    errors = []

    def validate_value(value, schema, path="root"):
        schema_type = schema.get("type")

        # Type checking
        type_map = {
            "string": str, "integer": int, "number": (int, float),
            "boolean": bool, "array": list, "object": dict,
        }
        if schema_type in type_map:
            expected = type_map[schema_type]
            if not isinstance(value, expected):
                errors.append(f"{path}: expected {schema_type}, got {type(value).__name__}")
                return

        # Enum validation
        if "enum" in schema and value not in schema["enum"]:
            errors.append(f"{path}: '{value}' not in {schema['enum']}")

        # Object validation
        if schema_type == "object" and isinstance(value, dict):
            # Check required fields
            for req in schema.get("required", []):
                if req not in value:
                    errors.append(f"{path}: missing required field '{req}'")

            # Validate properties
            for prop, prop_schema in schema.get("properties", {}).items():
                if prop in value:
                    validate_value(value[prop], prop_schema, f"{path}.{prop}")

        # Array validation
        if schema_type == "array" and isinstance(value, list):
            items_schema = schema.get("items", {})
            for i, item in enumerate(value):
                validate_value(item, items_schema, f"{path}[{i}]")

    validate_value(data, schema)
    return errors


# Define the review output schema
review_schema = json_review.json_schema

# Test with a VALID review output
valid_output = {
    "issues": [
        {
            "file": "src/api/users.ts",
            "line": 42,
            "severity": "critical",
            "description": "SQL injection vulnerability: user input concatenated into query",
            "suggestion": "Use parameterized queries with $1 placeholders"
        },
        {
            "file": "src/api/users.ts",
            "line": 78,
            "severity": "warning",
            "description": "Missing null check on user.email before toLowerCase()",
            "suggestion": "Add: if (!user.email) return null;"
        },
        {
            "file": "src/components/Dashboard.tsx",
            "line": 15,
            "severity": "info",
            "description": "Consider memoizing this expensive computation",
            "suggestion": "Wrap in useMemo(() => computeStats(data), [data])"
        }
    ],
    "summary": "Found 1 critical security issue (SQL injection), 1 potential null pointer, and 1 performance suggestion.",
    "approve": False
}

errors = validate_against_schema(valid_output, review_schema)
print("Valid output validation:")
if not errors:
    print("  ✅ No errors — output conforms to schema!")
else:
    for e in errors:
        print(f"  ❌ {e}")

# Test with an INVALID output (missing required field, wrong enum)
invalid_output = {
    "issues": [
        {
            "file": "src/api/users.ts",
            # "line" is missing (required!)
            "severity": "extreme",  # Not in enum!
            "description": "Bad code"
        }
    ],
    # "summary" is missing (required!)
    "approve": "yes"  # Wrong type! Should be boolean
}

print("\nInvalid output validation:")
errors = validate_against_schema(invalid_output, review_schema)
for e in errors:
    print(f"  ❌ {e}")

In [ ]:
#@title 🎧 Listen: Pipeline Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_08_pipeline_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Building a Mock PR Review Pipeline

Let us simulate a complete CI/CD review pipeline.

In [ ]:
#@title 🎧 Listen: Pipeline Walkthrough
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_09_pipeline_walkthrough.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
@dataclass
class PRDiff:
    """Simulates a PR diff with changed files."""
    pr_number: int
    branch: str
    files_changed: List[Dict[str, Any]]
    prior_findings: List[Dict[str, str]] = field(default_factory=list)


@dataclass
class ReviewResult:
    """Structured review result from Claude Code."""
    issues: List[Dict[str, Any]]
    summary: str
    approve: bool
    session_id: str  # For tracking session isolation


def build_review_prompt(diff: PRDiff, claude_md_context: str) -> str:
    """
    Build the review prompt for Claude Code in CI mode.
    Includes CLAUDE.md context and prior findings for deduplication.
    """
    prompt_parts = [
        f"Review PR #{diff.pr_number} (branch: {diff.branch}).",
        "",
        "## Project Conventions (from CLAUDE.md)",
        claude_md_context,
        "",
        "## Changed Files",
    ]

    for file in diff.files_changed:
        prompt_parts.append(f"### {file['path']} (+{file['additions']}/-{file['deletions']})")
        prompt_parts.append(f"```\n{file['diff_snippet']}\n```")

    # Include prior findings to avoid duplicates
    if diff.prior_findings:
        prompt_parts.append("")
        prompt_parts.append("## Prior Review Findings (already addressed — do NOT re-flag)")
        for finding in diff.prior_findings:
            prompt_parts.append(f"- {finding['file']}:{finding['line']}: {finding['description']} ({finding['status']})")

    prompt_parts.append("")
    prompt_parts.append("Focus on NEW issues only. Output structured JSON per the schema.")

    return "\n".join(prompt_parts)


# Simulate a PR
pr = PRDiff(
    pr_number=456,
    branch="feature/user-preferences",
    files_changed=[
        {
            "path": "src/api/preferences.ts",
            "additions": 45,
            "deletions": 3,
            "diff_snippet": "async function updatePreferences(req, res) {\n"
                           "  const prefs = req.body;  // No validation!\n"
                           "  await db.query(`UPDATE prefs SET data = '${prefs}'`);\n"
                           "  res.json({ success: true });\n}"
        },
        {
            "path": "src/api/preferences.test.ts",
            "additions": 20,
            "deletions": 0,
            "diff_snippet": "describe('preferences', () => {\n"
                           "  it('updates preferences', async () => {\n"
                           "    // Only happy path tested\n"
                           "  });\n});"
        }
    ],
    prior_findings=[
        {"file": "src/api/preferences.ts", "line": "12", "description": "Missing type annotations", "status": "FIXED"},
    ]
)

claude_md = """- Framework: Express with TypeScript
- Validation: Zod schemas on all API inputs
- SQL: Parameterized queries only (never string concatenation)
- Testing: Vitest, minimum 80% coverage, test error paths"""

prompt = build_review_prompt(pr, claude_md)
print("Generated CI Review Prompt:")
print("=" * 60)
print(prompt)

In [ ]:
#@title 🎧 Listen: Session Isolation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_10_session_isolation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Session Context Isolation

This is a subtle but critical point: **Claude reviewing its own output in the same session is biased**. The exam tests this.

In [ ]:
def demonstrate_session_isolation():
    """Show why fresh sessions are important for CI review."""

    print("Session Context Isolation")
    print("=" * 60)

    print("""
❌ SAME SESSION (biased):
┌─────────────────────────────────────────┐
│ Session A                               │
│                                         │
│ [Step 1] Claude writes the code         │
│ [Step 2] Claude reviews its own code    │
│                                         │
│ Problem: Claude has memory of writing   │
│ the code. It "knows" what it intended,  │
│ so it's less likely to spot issues.     │
│ The code "looks right" because Claude   │
│ just wrote it.                          │
└─────────────────────────────────────────┘

✅ SEPARATE SESSION (unbiased):
┌──────────────────────┐   ┌──────────────────────┐
│ Session A            │   │ Session B (CI)       │
│                      │   │                      │
│ Developer + Claude   │   │ claude -p "Review    │
│ write the code       │   │   this PR..." \     │
│                      │   │   --output-format    │
│ Push PR →            │──>│   json               │
│                      │   │                      │
│                      │   │ Fresh context.       │
│                      │   │ No memory of writing │
│                      │   │ the code. Unbiased.  │
└──────────────────────┘   └──────────────────────┘
""")

    print("Key insight for the exam:")
    print("  • CI reviews should ALWAYS use a fresh session (-p flag)")
    print("  • Never reuse an interactive session for automated review")
    print("  • This is analogous to a different person reviewing your code")


demonstrate_session_isolation()

In [ ]:
#@title 🎧 Listen: Full Workflow
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_11_full_workflow.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Complete CI Workflow Simulation

Let us put everything together into a complete GitHub Actions-style workflow.

In [ ]:
def simulate_ci_workflow(pr: PRDiff) -> Dict[str, Any]:
    """Simulate a complete CI review workflow."""
    workflow_log = []

    # Step 1: Trigger
    workflow_log.append({
        "step": "trigger",
        "event": f"PR #{pr.pr_number} updated",
        "branch": pr.branch,
        "timestamp": datetime.now().isoformat()
    })

    # Step 2: Build the review command
    claude_md = "framework: Express+TS, validation: Zod, sql: parameterized, tests: Vitest 80%"
    prompt = build_review_prompt(pr, claude_md)

    cli_command = f"""claude -p "{prompt[:50]}..." \\
  --output-format json \\
  --json-schema '<review_schema>'"""

    workflow_log.append({
        "step": "invoke_claude",
        "command": cli_command,
        "session": "fresh (isolated)",
    })

    # Step 3: Simulate Claude's response
    review_output = {
        "issues": [
            {
                "file": "src/api/preferences.ts",
                "line": 3,
                "severity": "critical",
                "description": "SQL injection: string interpolation in query",
                "suggestion": "Use parameterized query: db.query('UPDATE prefs SET data = $1', [prefs])"
            },
            {
                "file": "src/api/preferences.ts",
                "line": 2,
                "severity": "warning",
                "description": "No input validation — req.body is unvalidated",
                "suggestion": "Add Zod schema: const PrefsSchema = z.object({...}); PrefsSchema.parse(req.body)"
            },
            {
                "file": "src/api/preferences.test.ts",
                "line": 3,
                "severity": "warning",
                "description": "Only happy path tested — missing error path tests",
                "suggestion": "Add tests for: invalid input, SQL errors, missing auth"
            }
        ],
        "summary": "1 critical SQL injection, 2 warnings (missing validation, incomplete tests)",
        "approve": False
    }

    # Step 4: Validate output against schema
    errors = validate_against_schema(review_output, review_schema)
    workflow_log.append({
        "step": "validate_output",
        "schema_valid": len(errors) == 0,
        "errors": errors,
    })

    # Step 5: Post PR comments
    comments_posted = []
    for issue in review_output["issues"]:
        comments_posted.append({
            "file": issue["file"],
            "line": issue["line"],
            "body": f"**[{issue['severity'].upper()}]** {issue['description']}\n\n"
                    f"💡 Suggestion: {issue.get('suggestion', 'N/A')}",
        })

    workflow_log.append({
        "step": "post_comments",
        "comments_count": len(comments_posted),
        "comments": comments_posted,
    })

    # Step 6: Set merge status
    workflow_log.append({
        "step": "set_status",
        "approve": review_output["approve"],
        "status": "blocked" if not review_output["approve"] else "approved",
        "reason": review_output["summary"],
    })

    return {
        "workflow_log": workflow_log,
        "review": review_output,
        "comments": comments_posted,
    }


result = simulate_ci_workflow(pr)

print("CI/CD Review Workflow Execution Log")
print("=" * 60)
for entry in result["workflow_log"]:
    step = entry["step"]
    print(f"\n📍 Step: {step}")
    for k, v in entry.items():
        if k != "step":
            if isinstance(v, list) and len(v) > 2:
                print(f"   {k}: [{len(v)} items]")
            else:
                print(f"   {k}: {v}")

print("\n" + "=" * 60)
print("Posted PR Comments:")
for comment in result["comments"]:
    print(f"\n  📝 {comment['file']}:{comment['line']}")
    print(f"     {comment['body'][:100]}...")

merge_status = result["workflow_log"][-1]
status_emoji = "🚫" if merge_status["status"] == "blocked" else "✅"
print(f"\n{status_emoji} Merge status: {merge_status['status'].upper()}")
print(f"   Reason: {merge_status['reason']}")

In [ ]:
#@title 🎧 Listen: Todo Schema
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_12_todo_schema.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. 🔧 Your Turn — Design a CI Review Schema

In [ ]:
# ============ TODO ============
# Design a JSON schema for a SECURITY-FOCUSED CI review.
# Your schema should capture:
# 1. A list of security findings (each with: file, line, vulnerability_type,
#    severity [critical/high/medium/low], description, cwe_id)
# 2. An overall risk_score (integer 0-100)
# 3. A list of positive_patterns (good security practices found)
# 4. A boolean block_merge flag

security_review_schema = {
    "type": "object",
    "properties": {
        # YOUR SCHEMA HERE
    },
    "required": []  # FILL IN REQUIRED FIELDS
}
# ==============================

In [ ]:
# ✅ Verification — test your schema with sample data
sample_security_review = {
    "findings": [
        {
            "file": "src/api/auth.ts",
            "line": 23,
            "vulnerability_type": "sql_injection",
            "severity": "critical",
            "description": "User input in SQL query without parameterization",
            "cwe_id": "CWE-89"
        }
    ],
    "risk_score": 85,
    "positive_patterns": [
        "HTTPS enforced on all endpoints",
        "Rate limiting configured on auth routes"
    ],
    "block_merge": True
}

# Check if the schema has the right structure
required_top_level = {"findings", "risk_score", "positive_patterns", "block_merge"}
schema_props = set(security_review_schema.get("properties", {}).keys())
schema_required = set(security_review_schema.get("required", []))

if required_top_level.issubset(schema_props):
    print("✅ Schema has all required top-level properties!")
else:
    missing = required_top_level - schema_props
    print(f"❌ Missing properties: {missing}")

if required_top_level.issubset(schema_required):
    print("✅ All key fields are marked as required!")
else:
    missing = required_top_level - schema_required
    print(f"⚠️ These should probably be required: {missing}")

# Validate sample data against the student's schema
errors = validate_against_schema(sample_security_review, security_review_schema)
if not errors:
    print("✅ Sample data validates against your schema!")
else:
    for e in errors:
        print(f"❌ {e}")

In [ ]:
#@title 🎧 Listen: Claude Md Ci
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_13_claude_md_ci.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. CLAUDE.md for CI — What to Document

Your CLAUDE.md should include CI-relevant information that shapes automated reviews.

In [ ]:
ci_claude_md_template = """
# CI/CD Review Configuration

## How to Run Tests
- Unit tests: `npm run test`
- Integration tests: `npm run test:integration`
- Type checking: `npx tsc --noEmit`

## Test Coverage Requirements
- Minimum: 80% line coverage
- All new API endpoints must have error path tests
- All new components must have snapshot tests

## Security Standards
- No string concatenation in SQL queries
- All user input must be validated with Zod
- No secrets in code (use environment variables)
- CORS must be explicitly configured per endpoint

## Code Conventions
- TypeScript strict mode (no `any` types)
- All functions must have explicit return types
- Error responses must use the standard envelope format
- All API routes must use the withErrorHandler middleware

## Review Focus Areas
- Type safety and null safety
- Error handling completeness
- Test coverage for new code
- Security vulnerabilities (OWASP Top 10)
- API contract backwards compatibility
"""

print("CLAUDE.md Template for CI/CD Reviews:")
print("=" * 60)
print(ci_claude_md_template)
print("\nThis gives Claude all the context it needs for automated reviews.")
print("Without it, Claude uses generic best practices that may")
print("conflict with your team's specific conventions.")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/04_14_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection and Next Steps

**Key takeaways:**
1. **`-p` flag**: Non-interactive mode for CI pipelines. No human at the terminal.
2. **`--output-format json`**: Machine-parseable output for downstream tools.
3. **`--json-schema`**: Forces Claude's output to conform to your schema.
4. **Session isolation**: Fresh sessions for unbiased reviews. Never reuse interactive sessions.
5. **Prior findings**: Include previously-addressed issues to avoid duplicate flagging.
6. **CLAUDE.md in CI**: Document test commands, coverage requirements, and conventions.

**Exam tip:** The exam often asks about the *why* behind session isolation and structured output. Understand the reasoning, not just the flags.

**What to try next:**
- Add a Claude Code review step to your actual CI pipeline
- Create a custom review schema for your project's specific needs
- Document your testing standards in CLAUDE.md